# Plotly Graph Objects & Dashboards: Beginner Guide
Learn low-level control with `plotly.graph_objects`: Custom Traces, Multi-Chart Dashboards, Dark Themes, and HTML Hover Popups.

### 📚 What You Will Learn in this Guide:
- **Low-Level Traces (`go.Scatter`, `go.Bar`, `go.Pie`)**: How to manually build and stack custom layers.
- **Multi-Chart Dashboards (`make_subplots`)**: How to combine separate charts into professional multi-panel layouts.
- **Global Layout Styling (`fig.update_layout`)**: How to apply dark mode themes and custom fonts.
- **Custom HTML Tooltips (`hovertemplate`)**: How to format popup boxes with custom text and currencies.

> **💡 Beginner Note**: Every single concept is isolated in its own section with:
> 1. **What is this?** (Plain English explanation)
> 2. **Why do we use it?** (Real-world intuition)
> 3. **Syntax & Parameters** (Parameter-by-parameter breakdown)
> 4. **Live Python Code** with outputs using `data/raw_transactions.csv`.

In [1]:
# Step 1: Import all necessary libraries
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Step 2: Set clean visual defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 10

# Step 3: Load the transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)

# Step 4: Ensure dates and numeric values are clean
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce')
df['account_age_months'] = pd.to_numeric(df['account_age_months'], errors='coerce')
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)
df = df.dropna(subset=['transaction_amount', 'transaction_date']).reset_index(drop=True)

print(f"✅ Successfully loaded {len(df)} transactions from {csv_path}")
df.head(3)

✅ Successfully loaded 14262 transactions from data/raw_transactions.csv


## 🔹 Low-Level Scatter & Line Traces: `go.Scatter()`

### 1. What is this?
`go.Scatter()` gives you 100% manual control over marker shapes, line styles, and colors.

### 2. Why do we use it?
Use Graph Objects (`go`) when you need fine-grained control that Plotly Express doesn't expose.

### 3. Syntax & Parameters
```python
trace = go.Scatter(x=..., y=..., mode='lines+markers', marker=dict(color='blue'))
fig = go.Figure(data=[trace])
```

In [2]:
scat_trace = go.Scatter(
    x=df['account_age_months'].head(50), 
    y=df['transaction_amount'].head(50), 
    mode='markers', 
    marker=dict(size=9, color='#1f77b4', symbol='circle'), 
    name='Transactions'
)

fig = go.Figure(data=[scat_trace], layout=go.Layout(title='Low-Level go.Scatter Layer'))
fig.show()

Figure(Low-Level go.Scatter Layer)

## 🔹 Pie and Donut Breakdown Charts: `go.Pie()`

### 1. What is this?
`go.Pie(hole=0.4)` draws a clean circular donut breakdown chart showing percentage proportions.

### 2. Why do we use it?
Standard for market share, category splits, and portfolio breakdowns.

### 3. Syntax & Parameters
```python
pie_trace = go.Pie(labels=categories, values=counts, hole=0.45)
```

In [3]:
card_counts = df['card_type'].value_counts()

pie_trace = go.Pie(labels=card_counts.index, values=card_counts.values, hole=0.45, hoverinfo='label+percent+value')
fig = go.Figure(data=[pie_trace], layout=go.Layout(title='Card Issuer Proportions (go.Pie Donut Chart)'))
fig.show()

Figure(Card Issuer Proportions (go.Pie Donut Chart))

## 🔹 Multi-Chart Dashboards: `make_subplots()`

### 1. What is this?
`make_subplots()` combines different chart types (like a bar chart and a donut chart) into a single unified dashboard layout.

### 2. Why do we use it?
Essential for executive dashboards and monitoring screens.

### 3. Syntax & Parameters
```python
fig = make_subplots(rows=1, cols=2, subplot_titles=('Chart 1', 'Chart 2'))
fig.add_trace(trace1, row=1, col=1)
fig.add_trace(trace2, row=1, col=2)
```

In [4]:
reg_data = df.groupby('region')['transaction_amount'].sum()
bar_trace = go.Bar(x=reg_data.index, y=reg_data.values, marker_color='teal', name='Revenue ($)')

fig = make_subplots(rows=1, cols=2, subplot_titles=('Regional Revenue', 'Card Market Share'))
fig.add_trace(bar_trace, row=1, col=1)
fig.add_trace(pie_trace, row=1, col=2)

fig.update_layout(title_text='Multi-Chart Analytics Dashboard (make_subplots)')
fig.show()

Traceback (most recent call last):
  File "C:\Users\DELL\investigate-pandas\scratch\render_all_beginner_notebooks.py", line 64, in render_notebook
    exec(code, exec_globals)
  File "<string>", line 6, in <module>
  File "C:\Users\DELL\anaconda3\Lib\site-packages\plotly\graph_objs\_figure.py", line 917, in add_trace
    return super(Figure, self).add_trace(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\plotly\basedatatypes.py", line 2106, in add_trace
    return self.add_traces(
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\plotly\graph_objs\_figure.py", line 997, in add_traces
    return super(Figure, self).add_traces(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\plotly\basedatatypes.py", line 2236, in add_traces
    self._set_trace_grid_position(trace, row, col, secondary_y)
  File "C:\Users\DELL\anaconda3\Lib\site-packages\plotly\basedatatypes.py", line 2328

## 🔹 Dark Theme & Global Styling: `fig.update_layout()`

### 1. What is this?
`update_layout()` changes properties across the entire chart at once (such as enabling the dark mode template or changing fonts).

### 2. Why do we use it?
Instantly make your charts look sleek and modern.

### 3. Syntax & Parameters
```python
fig.update_layout(template='plotly_dark', font=dict(family='Arial'))
```

In [5]:
fig = go.Figure(data=[bar_trace])
fig.update_layout(template='plotly_dark', title='Sleek Dark Theme Applied via update_layout()')
fig.show()

Figure(Sleek Dark Theme Applied via update_layout())

## 🔹 Custom HTML Hover Popups: `hovertemplate`

### 1. What is this?
`hovertemplate` lets you design the exact HTML layout of the popup box that appears when hovering over a point.

### 2. Why do we use it?
Format currencies cleanly (e.g. `$1,250.00`) and display custom bold labels.

### 3. Syntax & Parameters
```python
trace.hovertemplate = '<b>ID:</b> %{x}<br><b>Amount:</b> $%{y:,.2f}<extra></extra>'
```

In [6]:
custom_trace = go.Scatter(
    x=df['transaction_id'].head(10), 
    y=df['transaction_amount'].head(10), 
    mode='markers+lines', 
    hovertemplate='<b>Transaction ID:</b> %{x}<br><b>Spend:</b> $%{y:,.2f}<extra></extra>'
)

fig = go.Figure(data=[custom_trace], layout=go.Layout(title='Custom HTML Hovertemplate Formatting (Hover over points!)'))
fig.show()

Figure(Custom HTML Hovertemplate Formatting (Hover over points!))